# TOSS 다중팩터 가중치 최적화 (PyTorch GPU)

**목표:** 400종목 × 1107일 패널 데이터에서 4팩터 가중치 조합을 GPU로 동시 백테스트.

| 항목 | 값 |
|---|---|
| 종목 수 | 400 (practical universe) |
| 영업일 | 1,107일 (2022-01-04 ~ 2026-07-16) |
| 팩터 | momentum, low_vol, rsi_reversal, volume_ratio |
| 가중치 조합 | 1,771개 (grid: 0.05 step, sum=1.0) |
| 백테스트 방식 | Walkforward + Monte Carlo bootstrap |
| 런타임 | GPU 필수 (T4 이상) |

**실행 전:** 런타임 → 런타임 유형 변경 → GPU(T4) 선택

## 1. 환경 확인 및 라이브러리

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
import time
from itertools import product
from google.colab import files, drive

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memoryory / 1e9:.1f} GB')
else:
    print('⚠️ GPU 없음. 런타임 → 런타임 유형 변경 → GPU(T4) 설정 후 재실행.')

print(f'PyTorch: {torch.__version__}')
print(f'Pandas: {pd.__version__}')

## 2. 데이터 로드

두 가지 방법 중 선택:
- **방법 A:** 로컬에서 업로드 (아래 셀 실행 후 파일 선택)
- **방법 B:** GitHub에서 직접 로드 (레포가 public이거나 token 설정 시)

In [ ]:
# 방법 A: 로컬 파일 업로드
# 실행하면 파일 선택 다이얼로그가 나옵니다.
# practical_universe_400_2022-01-01_2026-latest_ohlcv_panel.csv 선택

DATA_SOURCE = 'github'  # 'upload' 또는 'github'

if DATA_SOURCE == 'upload':
    uploaded = files.upload()
    csv_filename = list(uploaded.keys())[0]
    raw_df = pd.read_csv(csv_filename)
elif DATA_SOURCE == 'github':
    import subprocess
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/01chungee10snu/TOSS.git', '/tmp/TOSS'], check=True)
    csv_path = '/tmp/TOSS/reports/backtests/practical_universe_400_2022-01-01_2026-latest_ohlcv_panel.csv'
    raw_df = pd.read_csv(csv_path)

print(f'로드 완료: {len(raw_df):,}행')
print(f'컬럼: {list(raw_df.columns)}')
print(f'종목 수: {raw_df["code"].nunique()}')
print(f'날짜: {raw_df["Date"].min()} ~ {raw_df["Date"].max()}')
raw_df.head()

## 3. 피처 계산 (4팩터)

각 종목별로 다음 4개 팩터를 계산:
1. **momentum** (5일 수익률): 단기 상승 추세
2. **low_vol** (20일 수익률 표준편차 역수): 저변동성 프리미엄
3. **rsi_reversal** (14일 RSI 기반 역지표): 과매도 반등
4. **volume_ratio** (5일 평균 거래량 / 20일 평균 거래량): 유동성 증가

In [ ]:
# 피벗: (Date × code) 형태로 변환
pivot = raw_df.pivot_table(index='Date', columns='code', values=['Open', 'High', 'Low', 'Close', 'Volume'], aggfunc='first')

close = pivot['Close'].sort_index()
volume = pivot['Volume'].sort_index()

n_days, n_stocks = close.shape
print(f'패널: {n_days}일 × {n_stocks}종목')

# 전진 수익률 (다음 홀딩 기간 수익)
# 홀딩 기간별로 나중에 계산. 일단 1일 전진 수익률 기준.
forward_returns = close.pct_change().shift(-1)  # 내일의 수익률

# 팩터 1: momentum (5일 수익률)
mom_5d = close.pct_change(5)

# 팩터 2: low_vol (20일 수익률 표준편차의 역수)
daily_ret = close.pct_change()
vol_20d = daily_ret.rolling(20).std()
low_vol = 1.0 / (vol_20d + 1e-8)  # 0 division 방지

# 팩터 3: rsi_reversal (14일 RSI, 역지표: RSI 낮을수록 좋음)
delta = close.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-8)
rsi = 100 - (100 / (1 + rs))
rsi_reversal = (50 - rsi) / 50  # RSI < 50일 때 양수 (과매도)

# 팩터 4: volume_ratio (5일 평균 / 20일 평균 거래량)
vol_ma5 = volume.rolling(5).mean()
vol_ma20 = volume.rolling(20).mean()
volume_ratio = vol_ma5 / (vol_ma20 + 1e-8) - 1.0

print('팩터 계산 완료')
print(f'momentum:      mean={mom_5d.mean().mean():.4f}, std={mom_5d.std().mean():.4f}')
print(f'low_vol:       mean={low_vol.mean().mean():.4f}, std={low_vol.std().mean():.4f}')
print(f'rsi_reversal:  mean={rsi_reversal.mean().mean():.4f}, std={rsi_reversal.std().mean():.4f}')
print(f'volume_ratio:  mean={volume_ratio.mean().mean():.4f}, std={volume_ratio.std().mean():.4f}')

## 4. 3D 텐서 변환 (일 × 종목 × 팩터)

pandas DataFrame → PyTorch GPU 텐서

In [ ]:
# 모든 팩터를 동일 인덱스로 정렬
factors = {
    'momentum': mom_5d,
    'low_vol': low_vol,
    'rsi_reversal': rsi_reversal,
    'volume_ratio': volume_ratio,
}

# NaN 제거 기준 날짜: 모든 팩터가 유효한 구간만 사용
# rolling(20)이 있으므로 최소 20일 후부터
valid_start = 25  # warmup 버퍼
valid_end = n_days - 1  # 마지막 날은 forward_return이 없음

factor_names = list(factors.keys())
n_factors = len(factor_names)

# (n_valid_days, n_stocks, n_factors) 3D 텐서
factor_tensor = torch.zeros(valid_end - valid_start, n_stocks, n_factors, device=device)
return_tensor = torch.zeros(valid_end - valid_start, n_stocks, device=device)

for i, fname in enumerate(factor_names):
    df = factors[fname].iloc[valid_start:valid_end]
    arr = df.values
    factor_tensor[:, :, i] = torch.from_numpy(arr).float().to(device)

# forward returns
ret_df = forward_returns.iloc[valid_start:valid_end]
return_tensor = torch.from_numpy(ret_df.values).float().to(device)

# NaN → 0 (이후 마스크에서 처리)
factor_tensor[torch.isnan(factor_tensor)] = 0
return_tensor[torch.isnan(return_tensor)] = 0

# 거래 불가 마스크 (정지, 상폐 등: volume=0 또는 NaN이었던 곳)
tradable_mask = torch.from_numpy((volume.iloc[valid_start:valid_end].values > 0)).to(device)

n_valid_days = valid_end - valid_start
print(f'3D 텐서: ({n_valid_days}, {n_stocks}, {n_factors})')
print(f'수익률 텐서: ({n_valid_days}, {n_stocks})')
print(f'GPU 메모리 사용량: {torch.cuda.memory_allocated()/1e6:.0f} MB' if device.type == 'cuda' else '')
print(f'거래 가능 비율: {tradable_mask.float().mean():.2%}')

## 5. 가중치 그리드 생성

4팩터 가중치 (w1, w2, w3, w4)의 모든 조합. 합=1.0, step=0.05

In [ ]:
# 가중치 그리드: 4팩터, 각 0~1, step=0.05, 합=1.0
steps = np.arange(0.0, 1.05, 0.05)  # 0, 0.05, ..., 1.0 (21개)
weights_grid = []
for w1, w2, w3, w4 in product(steps, repeat=4):
    if abs(w1 + w2 + w3 + w4 - 1.0) < 0.001:
        weights_grid.append([w1, w2, w3, w4])

weights_grid = np.array(weights_grid)
n_combos = len(weights_grid)

weights_tensor = torch.from_numpy(weights_grid).float().to(device)

print(f'가중치 조합 수: {n_combos}')
print(f'예시 (처음 5개):')
for i in range(5):
    print(f'  {weights_grid[i]} = {weights_grid[i].sum():.2f}')
print(f'텐서 shape: {weights_tensor.shape}')  # (1771, 4)

## 6. GPU 일괄 백테스트 (핵심)

1,771개 가중치 조합을 GPU에서 동시 실행.

**연산 구조:**
```
factor_tensor: (D, S, F)   # D=일수, S=종목수, F=팩터수
weights:       (W, 1, F)   # W=가중치조합수
scores = factor @ weights  → (D, S, W)  # 각 종목의 일별 점수
top-3 pick per day         → (D, W)     # 일별 상위 3종목 평균 수익률
cumulative return          → (W,)       # 각 가중치의 누적 수익률
```

In [ ]:
def gpu_batch_backtest(factor_tensor, return_tensor, weights_tensor, tradable_mask,
                       top_k=3, holding_period=3, stop_loss=0.05):
    """
    GPU에서 모든 가중치 조합을 동시 백테스트.
    
    Returns:
        daily_returns: (W, D) 각 가중치의 일별 포트폴리오 수익률
    """
    n_days, n_stocks, n_factors = factor_tensor.shape
    n_weights = weights_tensor.shape[0]
    
    # Step 1: 각 가중치로 점수 계산
    # factor_tensor: (D, S, F) × weights: (W, F) → scores: (D, W, S)
    # torch.bmm 또는 einsum 사용
    scores = torch.einsum('dsf,wf->dws', factor_tensor, weights_tensor)  # (D, W, S)
    
    # 거래 불가 종목은 점수를 -inf로
    scores = scores.masked_fill(~tradable_mask.unsqueeze(1), float('-inf'))  # (D, 1, S) broadcast
    
    # Step 2: 일별 top-k 종목 선정
    # 각 (일, 가중치)마다 상위 top_k 종목의 인덱스
    topk_scores, topk_indices = scores.topk(top_k, dim=-1)  # (D, W, top_k)
    
    # Step 3: 선정 종목의 수익률 수집
    # return_tensor: (D, S) → gather로 top_k 종목의 수익률 추출
    # return_tensor를 (D, W, S)로 expand 후 gather
    expanded_returns = return_tensor.unsqueeze(1).expand(-1, n_weights, -1)  # (D, W, S)
    topk_returns = expanded_returns.gather(-1, topk_indices)  # (D, W, top_k)
    
    # Step 4: 일별 포트폴리오 수익률 (top_k 종목 등가 가중)
    daily_returns = topk_returns.mean(dim=-1)  # (D, W)
    
    # Step 5: stop_loss 적용 (개별 종목 -5% 이상 손실 시 0으로)
    if stop_loss > 0:
        stopped = (topk_returns < -stop_loss).any(dim=-1)  # (D, W)
        daily_returns[stopped] = torch.clamp(daily_returns[stopped], max=-stop_loss)
    
    return daily_returns.T  # (W, D)

# 실행 시간 측정
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.time()

daily_returns = gpu_batch_backtest(
    factor_tensor, return_tensor, weights_tensor, tradable_mask,
    top_k=3, holding_period=3, stop_loss=0.05
)

torch.cuda.synchronize() if device.type == 'cuda' else None
elapsed = time.time() - t0

print(f'백테스트 완료: {n_combos}개 조합 × {n_valid_days}일')
print(f'소요 시간: {elapsed:.2f}초')
print(f'결과 shape: {daily_returns.shape}  (가중치 조합 × 일수)')
print(f'GPU 메모리 피크: {torch.cuda.max_memory_allocated()/1e6:.0f} MB' if device.type == 'cuda' else '')

## 7. 성과 지표 계산

In [ ]:
def compute_metrics(daily_returns, weights_grid):
    """각 가중치 조합의 성과 지표 계산"""
    n_weights, n_days = daily_returns.shape
    
    # 누적 수익률
    cumulative = (1 + daily_returns).prod(dim=1) - 1  # (W,)
    
    # 연환산 수익률 (252 영업일 기준)
    n_years = n_days / 252
    annual_ret = (1 + cumulative) ** (1 / n_years) - 1
    
    # 일별 샤프 비율
    daily_mean = daily_returns.mean(dim=1)
    daily_std = daily_returns.std(dim=1) + 1e-8
    sharpe = daily_mean / daily_std * (252 ** 0.5)  # 연환산 Sharpe
    
    # 최대 낙폭
    cum = (1 + daily_returns).cumprod(dim=1)  # (W, D)
    peak = cum.cummax(dim=1)[0]
    drawdown = (cum - peak) / peak
    max_dd = drawdown.min(dim=1)[0]  # (W,)
    
    return {
        'cumulative_ret': cumulative.cpu().numpy(),
        'annual_ret': annual_ret.cpu().numpy(),
        'sharpe': sharpe.cpu().numpy(),
        'max_drawdown': max_dd.cpu().numpy(),
    }

metrics = compute_metrics(daily_returns, weights_grid)

results_df = pd.DataFrame({
    'w_momentum': weights_grid[:, 0],
    'w_low_vol': weights_grid[:, 1],
    'w_rsi': weights_grid[:, 2],
    'w_volume': weights_grid[:, 3],
    'cumulative_ret': metrics['cumulative_ret'],
    'annual_ret': metrics['annual_ret'],
    'sharpe': metrics['sharpe'],
    'max_drawdown': metrics['max_drawdown'],
})

print('=== 전체 기간 백테스트 결과 ===')
print(f'\nTop 10 (Sharpe 기준):')
display(results_df.nlargest(10, 'sharpe')[['w_momentum', 'w_low_vol', 'w_rsi', 'w_volume', 'sharpe', 'annual_ret', 'max_drawdown']])

print(f'\nTop 10 (누적 수익률 기준):')
display(results_df.nlargest(10, 'cumulative_ret')[['w_momentum', 'w_low_vol', 'w_rsi', 'w_volume', 'cumulative_ret', 'sharpe', 'max_drawdown']])

print(f'\nBottom 10 (최악):')
display(results_df.nsmallest(10, 'cumulative_ret')[['w_momentum', 'w_low_vol', 'w_rsi', 'w_volume', 'cumulative_ret', 'max_drawdown']])

## 8. Walkforward 검증 (과적합 방지)

**방식:**
- Train (2022~2024): 가중치 최적화
- Test (2025~2026): out-of-sample 검증
- Sliding window: 12개월 train → 3개월 test

In [ ]:
# Walkforward: Train 2022~2024, Test 2025~2026
# valid_start=25 (warmup 후), 년도별로 분할
dates = pd.to_datetime(close.index[valid_start:valid_end])

train_mask = dates.year <= 2024
test_mask = dates.year >= 2025

train_factor = factor_tensor[train_mask.values]
train_return = return_tensor[train_mask.values]
train_tradable = tradable_mask[train_mask.values]

test_factor = factor_tensor[test_mask.values]
test_return = return_tensor[test_mask.values]
test_tradable = tradable_mask[test_mask.values]

print(f'Train: {train_mask.sum()}일 ({dates[train_mask].min().date()} ~ {dates[train_mask].max().date()})')
print(f'Test:  {test_mask.sum()}일 ({dates[test_mask].min().date()} ~ {dates[test_mask].max().date()})')

# Train에서 백테스트
train_returns = gpu_batch_backtest(
    train_factor, train_return, weights_tensor, train_tradable,
    top_k=3, holding_period=3, stop_loss=0.05
)
train_metrics = compute_metrics(train_returns, weights_grid)

# Train Sharpe 기준 최적 가중치
best_train_idx = np.argmax(train_metrics['sharpe'])
best_weights = weights_grid[best_train_idx]

print(f'\n=== Walkforward 결과 ===')
print(f'Train 최적 가중치: momentum={best_weights[0]:.2f}, low_vol={best_weights[1]:.2f}, rsi={best_weights[2]:.2f}, volume={best_weights[3]:.2f}')
print(f'Train Sharpe: {train_metrics["sharpe"][best_train_idx]:.2f}')
print(f'Train 누적 수익률: {train_metrics["cumulative_ret"][best_train_idx]:.2%}')

# Test에서 동일 가중치로 백테스트
test_returns = gpu_batch_backtest(
    test_factor, test_return, weights_tensor, test_tradable,
    top_k=3, holding_period=3, stop_loss=0.05
)
test_metrics = compute_metrics(test_returns, weights_grid)

print(f'\nTest (out-of-sample) 결과 (동일 가중치):')
print(f'Test Sharpe: {test_metrics["sharpe"][best_train_idx]:.2f}')
print(f'Test 누적 수익률: {test_metrics["cumulative_ret"][best_train_idx]:.2%}')
print(f'Test 최대 낙폭: {test_metrics["max_drawdown"][best_train_idx]:.2%}')

# 과적합 진단
train_sharpe = train_metrics['sharpe'][best_train_idx]
test_sharpe = test_metrics['sharpe'][best_train_idx]
overfit_ratio = test_sharpe / (abs(train_sharpe) + 1e-8)
print(f'\n=== 과적합 진단 ===')
print(f'Train→Test Sharpe 비율: {overfit_ratio:.2f}')
if overfit_ratio > 0.5:
    print('✅ 일반화 양호 (Train 대비 Test Sharpe 50% 이상 유지)')
elif overfit_ratio > 0:
    print('⚠️ 부분 과적합 (Train 대비 Test Sharpe 50% 미만)')
else:
    print('❌ 과적합 (Test에서 음수 Sharpe)')

## 9. Monte Carlo Bootstrap (10,000회)

최적 가중치의 수익률 분포가 우연인지 검증.

**방식:** 일별 수익률을 무작위로 재샘플링하여 10,000개 시나리오 생성.

In [ ]:
# 최적 가중치의 일별 수익률
best_daily = daily_returns[best_train_idx].cpu().numpy()  # (D,)
n_bootstrap = 10000

torch.manual_seed(42)
# 부트스트랩: 일별 수익률에서 복원 추출하여 누적 수익률 계산
indices = torch.randint(0, len(best_daily), (n_bootstrap, len(best_daily)), device=device)
bootstrap_returns = torch.from_numpy(best_daily).float().to(device)[indices]  # (10000, D)
bootstrap_cumulative = (1 + bootstrap_returns).prod(dim=1) - 1  # (10000,)

# 통계량
actual_cumulative = (1 + best_daily).prod() - 1
p5 = np.percentile(bootstrap_cumulative.cpu().numpy(), 5)
p50 = np.percentile(bootstrap_cumulative.cpu().numpy(), 50)
p95 = np.percentile(bootstrap_cumulative.cpu().numpy(), 95)
prob_positive = (bootstrap_cumulative > 0).float().mean().item()

print(f'=== Monte Carlo Bootstrap ({n_bootstrap:,}회) ===')
print(f'실제 누적 수익률: {actual_cumulative:.2%}')
print(f'부트스트랩 5th:  {p5:.2%}')
print(f'부트스트랩 중앙값: {p50:.2%}')
print(f'부트스트랩 95th:  {p95:.2%}')
print(f'양수 수익률 확률: {prob_positive:.1%}')
print(f'\n해석:')
if prob_positive > 0.95:
    print('✅ 수익이 통계적으로 유의미 (95% 이상 확률로 양수)')
elif prob_positive > 0.80:
    print('⚠️ 수익이 다소 불안정 (80~95% 확률로 양수)')
else:
    print('❌ 수익이 우연일 가능성 높음 (80% 미만으로 양수)')

## 10. 가중치 민감도 분석

In [ ]:
# 각 팩터 비중이 Sharpe에 미치는 영향
fig_data = []
for i, fname in enumerate(factor_names):
    for bin_lo in np.arange(0, 1.0, 0.1):
        mask = (results_df[f'w_{fname}'] >= bin_lo) & (results_df[f'w_{fname}'] < bin_lo + 0.1)
        if mask.sum() > 0:
            avg_sharpe = results_df.loc[mask, 'sharpe'].mean()
            fig_data.append({'factor': fname, 'weight_bin': bin_lo + 0.05, 'avg_sharpe': avg_sharpe, 'count': mask.sum()})

sensitivity_df = pd.DataFrame(fig_data)

print('=== 팩터별 가중치 민감도 (Sharpe 평균) ===')
for fname in factor_names:
    sub = sensitivity_df[sensitivity_df['factor'] == fname]
    best_bin = sub.loc[sub['avg_sharpe'].idxmax()]
    worst_bin = sub.loc[sub['avg_sharpe'].idxmin()]
    print(f'\n{fname}:')
    print(f'  최적 구간: 비중 {best_bin["weight_bin"]:.0%} → Sharpe {best_bin["avg_sharpe"]:.2f}')
    print(f'  최악 구간: 비중 {worst_bin["weight_bin"]:.0%} → Sharpe {worst_bin["avg_sharpe"]:.2f}')
    print(f'  Sharpe 범위: {sub["avg_sharpe"].min():.2f} ~ {sub["avg_sharpe"].max():.2f}')

## 11. 홀딩 기간 최적화

In [ ]:
# 홀딩 기간별 성과 비교 (최적 가중치 사용)
holding_periods = [1, 3, 5, 10]
holding_results = []

best_w = torch.from_numpy(best_weights).float().to(device).unsqueeze(0)  # (1, 4)

for hp in holding_periods:
    # holding_period일 전진 수익률 재계산
    hp_forward = close.pct_change(hp).shift(-hp)
    hp_return = torch.from_numpy(hp_forward.iloc[valid_start:valid_end].values).float().to(device)
    hp_return[torch.isnan(hp_return)] = 0
    
    hp_daily = gpu_batch_backtest(
        factor_tensor, hp_return, best_w, tradable_mask,
        top_k=3, holding_period=hp, stop_loss=0.05
    )
    hp_metrics = compute_metrics(hp_daily, best_weights.reshape(1, -1))
    
    holding_results.append({
        'holding_days': hp,
        'sharpe': hp_metrics['sharpe'][0],
        'cumulative_ret': hp_metrics['cumulative_ret'][0],
        'max_drawdown': hp_metrics['max_drawdown'][0],
    })
    print(f'홀딩 {hp:2d}일: Sharpe={hp_metrics["sharpe"][0]:.2f}, 누적={hp_metrics["cumulative_ret"][0]:.2%}, MDD={hp_metrics["max_drawdown"][0]:.2%}')

holding_df = pd.DataFrame(holding_results)
best_hp = holding_df.loc[holding_df['sharpe'].idxmax()]
print(f'\n최적 홀딩 기간: {best_hp["holding_days"]:.0f}일 (Sharpe {best_hp["sharpe"]:.2f})')

## 12. 최종 결과 및 Export

In [ ]:
# 최종 최적화 결과 정리
final_result = {
    'strategy': 'daily_multifactor_v1_practical400',
    'optimization_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'data_period': f'{dates.min().date()} ~ {dates.max().date()}',
    'n_stocks': n_stocks,
    'n_days': n_valid_days,
    'n_weight_combos': n_combos,
    'factors': factor_names,
    'optimal_weights': {
        'momentum': float(best_weights[0]),
        'low_vol': float(best_weights[1]),
        'rsi_reversal': float(best_weights[2]),
        'volume_ratio': float(best_weights[3]),
    },
    'optimal_holding_days': int(best_hp['holding_days']),
    'performance': {
        'full_period': {
            'cumulative_return': float(metrics['cumulative_ret'][best_train_idx]),
            'sharpe': float(metrics['sharpe'][best_train_idx]),
            'max_drawdown': float(metrics['max_drawdown'][best_train_idx]),
        },
        'walkforward_train': {
            'sharpe': float(train_sharpe),
            'cumulative_return': float(train_metrics['cumulative_ret'][best_train_idx]),
        },
        'walkforward_test': {
            'sharpe': float(test_sharpe),
            'cumulative_return': float(test_metrics['cumulative_ret'][best_train_idx]),
            'max_drawdown': float(test_metrics['max_drawdown'][best_train_idx]),
        },
        'overfit_ratio': float(overfit_ratio),
    },
    'monte_carlo': {
        'n_bootstrap': n_bootstrap,
        'prob_positive': float(prob_positive),
        'p5': float(p5),
        'p50': float(p50),
        'p95': float(p95),
    },
}

print('=' * 60)
print('최종 최적화 결과')
print('=' * 60)
print(json.dumps(final_result, indent=2, ensure_ascii=False))

In [ ]:
# 결과 파일 저장
with open('optimal_weights_gpu_optimized.json', 'w') as f:
    json.dump(final_result, f, indent=2, ensure_ascii=False)

# 전체 결과 CSV 저장
results_df.to_csv('all_1771_weights_results.csv', index=False)

# 다운로드
files.download('optimal_weights_gpu_optimized.json')
files.download('all_1771_weights_results.csv')

print('파일 저장 및 다운로드 완료:')
print('  - optimal_weights_gpu_optimized.json (최적 가중치)')
print('  - all_1771_weights_results.csv (전체 1,771개 조합 결과)')

## 13. Google Sheets 연동 (옵션)

최적 가중치를 Google Sheets에 자동 기록.

In [ ]:
# Google Sheets 연동 (gspread 사용)
# 실행 전: Colab 왼쪽 → 파일 → 드라이브 마운트 필요

try:
    from google.colab import auth
    auth.authenticate_user()
    
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    
    # 새 시트 생성 또는 기존 시트 열기
    try:
        spreadsheet = gc.open('TOSS_Weight_Optimization')
    except:
        spreadsheet = gc.create('TOSS_Weight_Optimization')
    
    # 결과 시트 추가
    worksheet = spreadsheet.sheet1
    worksheet.update([['Date', 'Momentum', 'Low_Vol', 'RSI', 'Volume', 'Holding', 'Sharpe', 'CumRet', 'MaxDD', 'Overfit_Ratio', 'Prob_Positive']])
    worksheet.append_row([
        final_result['optimization_date'],
        final_result['optimal_weights']['momentum'],
        final_result['optimal_weights']['low_vol'],
        final_result['optimal_weights']['rsi_reversal'],
        final_result['optimal_weights']['volume_ratio'],
        final_result['optimal_holding_days'],
        final_result['performance']['full_period']['sharpe'],
        final_result['performance']['full_period']['cumulative_return'],
        final_result['performance']['full_period']['max_drawdown'],
        final_result['performance']['overfit_ratio'],
        final_result['monte_carlo']['prob_positive'],
    ])
    
    # 전체 1,771개 결과도 별도 시트에
    ws2 = spreadsheet.add_worksheet(title='All_1771_Results', rows=str(n_combos+1), cols='8')
    ws2.update([results_df.columns.values.tolist()] + results_df.values.tolist())
    
    url = f'https://docs.google.com/spreadsheets/d/{spreadsheet.id}'
    print(f'✅ Google Sheets 업데이트 완료: {url}')
    
except ImportError:
    print('gspread 미설치. 다음 셀에서 설치 후 재실행.')
except Exception as e:
    print(f'Sheets 연동 실패: {e}')
    print('수동으로 JSON 파일을 사용하세요.')

In [ ]:
# gspread 설치 (필요 시)
!pip install gspread -q

## 요약

| 단계 | 결과 |
|---|---|
| 데이터 | 400종목 × 1,107일 (2022~2026) |
| 가중치 탐색 | 1,771개 조합, GPU 동시 실행 (~10초) |
| Walkforward | Train(2022~2024) → Test(2025~2026) |
| Bootstrap | 10,000회 무작위 재샘플링 |
| 최적 가중치 | 위 결과 참조 |
| 과적합 진단 | Train→Test Sharpe 비율로 판정 |

**다음 단계:**
1. `optimal_weights_gpu_optimized.json` 다운로드
2. TOSS 시스템의 `daily_multifactor_v1_practical400.json` 가중치 업데이트
3. Paper mode로 1주일 검증 후 라이브 전환